# vLLM check (GPU): same 5 prompts with transformers and vLLM

Runs the Baseline 3 parser prompt (p1-v3) for 5 test tasks with each backend (each in its own process) and compares
the replies, the parsed contracts and the time per prompt (`eval/ext_vllm_compare.py`). Use this before running any
system on vLLM: greedy outputs can differ, which is why vLLM runs are their own version.
Installing vLLM may replace Kaggle's torch build; this notebook is separate for that reason.

In [ ]:
REPO_URL = "https://github.com/srinivas0307/lma-group-project.git"   # <- your repo
BRANCH = "main"
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"; LOAD_IN_4BIT = True
N = 5
BACKEND = "vllm"
NEEDED = ["eval/ext_vllm_compare.py", "src/igt/llm/vllm_backend.py", "requirements-vllm.txt"]

In [ ]:
# ---------------- Get the code ----------------
import os, shutil, subprocess, time, json, collections, zipfile, datetime
url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    print("no GITHUB_TOKEN secret; cloning without a token (public repo)")
WORK = "/kaggle/working"
os.chdir(WORK)
shutil.rmtree(f"{WORK}/repo", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, f"{WORK}/repo"], check=True)
os.chdir(f"{WORK}/repo")
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
missing = [f for f in NEEDED if not os.path.exists(f)]
assert not missing, f"missing (push them first): {missing}"
print("commit", COMMIT)

In [ ]:
# ---------------- Libraries: pinned pandas for the sandbox / verifier / tools (own folder) + bitsandbytes ----------------
subprocess.run(["pip", "install", "-q", "--target", f"{WORK}/evalpkgs", "pandas==2.3.3", "numpy==2.1.3",
                "pyyaml==6.0.2", "pydantic==2.14.1"], check=True)
subprocess.run(["pip", "install", "-q", "bitsandbytes"], check=True)
if BACKEND == "vllm":
    subprocess.run(["pip", "install", "-q", "-r", "requirements-vllm.txt"], check=True)
ENV = dict(os.environ, PYTHONPATH=f"{WORK}/evalpkgs")
# pip may print "ERROR: pip's dependency resolver ..." about Kaggle's pre-installed packages: only a warning.
probe = ("import pandas, numpy, torch, transformers; print('pandas', pandas.__version__, '| numpy', numpy.__version__, "
         "'| torch', torch.__version__, '| GPU', torch.cuda.is_available())")
p = subprocess.run(["python", "-c", probe], env=ENV, capture_output=True, text=True)
print(p.stdout, p.stderr[-1500:])
assert p.returncode == 0 and "pandas 2.3.3" in p.stdout and "GPU True" in p.stdout, "environment check failed"

In [ ]:
def sh(cmd, log=None):
    """Run a step, stream its output live, keep the full log."""
    t0 = time.time()
    p = subprocess.Popen(cmd, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in p.stdout:
        lines.append(line)
        if "Loading weights" not in line:
            print(line, end="", flush=True)
    rc = p.wait()
    if log:
        os.makedirs(os.path.dirname(log), exist_ok=True)
        open(log, "a").write(f"$ {' '.join(cmd)}\n" + "".join(lines) + f"\n--- exit {rc}, {(time.time() - t0) / 60:.1f} min ---\n")
    assert rc == 0, f"failed: {' '.join(cmd)}"
    return (time.time() - t0) / 60

for be in ("hf", "vllm"):
    cmd = ["python", "eval/ext_vllm_compare.py", "--backend", be, "--model", MODEL_ID, "--n", str(N),
           "--out", f"{WORK}/compare_{be}.json"] + (["--load-in-4bit"] if LOAD_IN_4BIT else [])
    print(f"== {be}: {sh(cmd):.1f} min")
sh(["python", "eval/ext_vllm_compare.py", "--compare", f"{WORK}/compare_hf.json", f"{WORK}/compare_vllm.json"])